# Results Visualization

This notebook creates figures from the saved intrusion detection results. It covers target adaptation, attack-category recall, model compression, and the frozen CIC holdout.

The figures use recorded metrics. No models are trained or evaluated here.

## 1. Load results

The figures use CSV and JSON files saved during the experiments. Each figure is exported as PNG, PDF, and SVG.

In [ ]:
from pathlib import Path
import os
import json
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Wedge, PathPatch
from matplotlib.path import Path as MplPath
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE = Path(os.environ.get('CSC786_DATA_ROOT', '/content/drive/MyDrive/datanew/a4_analysis'))
OUT = BASE / 'reproduction_runs' / 'visualizations'
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})

PALETTE = ['#4263A7', '#C5653A', '#289C8E', '#8357AC', '#C39A39', '#D8546A']
SHORT = {
    'Brute Force -Web': 'Web brute force',
    'Brute Force -XSS': 'XSS brute force',
    'DDOS attack-HOIC': 'DDoS HOIC',
    'DDOS attack-LOIC-UDP': 'DDoS LOIC UDP',
    'DDoS attacks-LOIC-HTTP': 'DDoS LOIC HTTP',
    'DoS attacks-GoldenEye': 'DoS GoldenEye',
    'DoS attacks-Hulk': 'DoS Hulk',
    'DoS attacks-SlowHTTPTest': 'DoS SlowHTTP',
    'DoS attacks-Slowloris': 'DoS Slowloris',
    'FTP-BruteForce': 'FTP brute force',
    'SSH-Bruteforce': 'SSH brute force',
    'Infilteration': 'Infilteration',
}

def locate(*paths):
    for rel in paths:
        candidate = BASE / rel
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('Missing result file. Checked:\n' + '\n'.join(str(BASE / p) for p in paths))

def check_columns(df, names, filename):
    missing = set(names).difference(df.columns)
    if missing:
        raise ValueError(f'{filename}: missing columns {sorted(missing)}')

def export(fig, name):
    for ext in ('png', 'pdf', 'svg'):
        fig.savefig(OUT / f'{name}.{ext}', dpi=320, bbox_inches='tight', facecolor='white')
    print('Saved:', name)

ADAPTATION_PATH = locate(
    'reproduction_runs/source_adaptation/adaptation_three_seed_results.csv',
    'cell_44K_three_method_comparison.csv'
)
ADAPTATION = pd.read_csv(ADAPTATION_PATH)
check_columns(ADAPTATION, ['Seed','Method','Macro_F1','Binary_PR_AUC'], ADAPTATION_PATH.name)

DEVELOPMENT_CATEGORY = BASE / 'reproduction_runs/model_compression/cic_development_category_recall.csv'
if DEVELOPMENT_CATEGORY.is_file():
    CATEGORY = pd.read_csv(DEVELOPMENT_CATEGORY)
    category_source = DEVELOPMENT_CATEGORY.name
else:
    pruning_path = locate('cell_44AB_pruning_category_recall.csv')
    distilled_path = locate('cell_44AD_distillation_category_results.csv')
    CATEGORY = pd.concat([pd.read_csv(pruning_path), pd.read_csv(distilled_path)], ignore_index=True)
    category_source = f'{pruning_path.name} + {distilled_path.name}'
check_columns(CATEGORY, ['Model','Attack_Category','Support','Recall'], category_source)
CATEGORY['Support'] = pd.to_numeric(CATEGORY['Support'])
CATEGORY['Recall'] = pd.to_numeric(CATEGORY['Recall'])
if CATEGORY.duplicated(['Model','Attack_Category']).any():
    raise ValueError('Repeated model–category combinations in recall table')
if not CATEGORY['Recall'].between(0,1).all():
    raise ValueError('Recall values must be between 0 and 1')

COMPRESSION_PATH = locate(
    'reproduction_runs/model_compression/final_compression_comparison.csv',
    'cell_44AE_final_compression_comparison.csv'
)
COMPRESSION = pd.read_csv(COMPRESSION_PATH)
check_columns(COMPRESSION, ['Model','Macro_F1','Binary_PR_AUC','Deployment_Artifacts_KB','Mean_Total_ms'], COMPRESSION_PATH.name)
if 'P95_Total_ms' not in COMPRESSION:
    latency_path = locate('cell_44AE_compression_latency.csv',
                          'reproduction_runs/model_compression/cpu_inference_latency.csv')
    latency = pd.read_csv(latency_path)
    check_columns(latency, ['Model','Batch_Size','P95_Total_ms'], latency_path.name)
    COMPRESSION = COMPRESSION.merge(
        latency.loc[latency['Batch_Size'] == 256, ['Model','P95_Total_ms']],
        on='Model', how='left', validate='one_to_one'
    )
check_columns(COMPRESSION, ['P95_Total_ms'], COMPRESSION_PATH.name)
if COMPRESSION['P95_Total_ms'].isna().any():
    raise ValueError('Batch-256 P95 latency is missing for some models')

HOLDOUT_CATEGORY_PATH = locate('cell_44R_category_results.csv',
    'reproduction_runs/evaluation_results/frozen_holdout_category_results.csv')
HOLDOUT_SUMMARY_PATH = locate('cell_44R_holdout_summary.json')
HOLDOUT_CATEGORY = pd.read_csv(HOLDOUT_CATEGORY_PATH)
check_columns(HOLDOUT_CATEGORY, ['Attack_Category','Test_Records','Recall'], HOLDOUT_CATEGORY_PATH.name)
with HOLDOUT_SUMMARY_PATH.open(encoding='utf-8') as file:
    HOLDOUT = json.load(file)
if int(HOLDOUT['Holdout_Records']) != 20000:
    raise ValueError('Unexpected frozen holdout size')
if int(HOLDOUT_CATEGORY['Test_Records'].sum()) != int(HOLDOUT['Attack_Records']):
    raise ValueError('Holdout category support does not match the summary')

print('Three-seed results:', ADAPTATION_PATH.name)
print('Attack-category results:', category_source)
print('Compression results:', COMPRESSION_PATH.name)
print('Holdout results:', HOLDOUT_SUMMARY_PATH.name)
print('Figures folder:', OUT)

## 2. Model–attack recognition network

Each center node is a model. The outer nodes are attack categories. A thicker connection means higher recall for that attack type, while larger outer nodes represent more evaluation records.

This figure uses the CIC development set.

In [ ]:
models = CATEGORY['Model'].drop_duplicates().tolist()
attack_order = (CATEGORY.groupby('Attack_Category')['Support'].max()
                .sort_values(ascending=False).index.tolist())

model_colors = {name: PALETTE[i % len(PALETTE)] for i, name in enumerate(models)}
lookup = CATEGORY.set_index(['Model','Attack_Category'])['Recall'].to_dict()
supports = CATEGORY.groupby('Attack_Category')['Support'].max().to_dict()

fig, ax = plt.subplots(figsize=(17, 14))
fig.patch.set_facecolor('white')
ax.set_aspect('equal')
ax.axis('off')

# Attack categories form the outer ring.
n_attack = len(attack_order)
theta = np.pi / 2 - np.arange(n_attack) * 2*np.pi/n_attack
attack_xy = {name: np.array([5.05*np.cos(t), 5.05*np.sin(t)])
             for name, t in zip(attack_order, theta)}

# Models are arranged near the center.
num_models = len(models)
model_xy = {}
for i, name in enumerate(models):
    angle = np.pi/2 - i*2*np.pi/num_models
    model_xy[name] = np.array([1.23*np.cos(angle), 1.23*np.sin(angle)])

outer = Circle((0,0),5.05, fill=False, edgecolor='#C8D6E2',lw=1.5,ls=':', zorder=0)
ax.add_patch(outer)

for i, model in enumerate(models):
    start = model_xy[model]
    for j, category in enumerate(attack_order):
        recall = float(lookup.get((model,category),0.0))
        if recall <= 0:
            continue
        end = attack_xy[category]
        angle = theta[j]
        # A small bend keeps the links visible without moving their endpoints.
        control1 = start*0.64 + end*0.27 + np.array([-np.sin(angle),np.cos(angle)])*.35
        control2 = start*0.25 + end*0.72 + np.array([-np.sin(angle),np.cos(angle)])*.18
        path = MplPath([start, control1, control2, end],
                       [MplPath.MOVETO,MplPath.CURVE4,MplPath.CURVE4,MplPath.CURVE4])
        ax.add_patch(PathPatch(path, facecolor='none',
                         edgecolor=model_colors[model],
                         lw=0.25+2.3*recall, alpha=0.10+0.34*recall,
                         zorder=1, capstyle='round'))

for model in models:
    xy = model_xy[model]
    ax.add_patch(Circle(xy,0.49,facecolor='white',
               edgecolor=model_colors[model],lw=3,zorder=5))
    label = model.replace('Teacher_', 'Teacher\n').replace('Distilled_','Distilled\n').replace('Pruned_','Pruned\n')
    label = label.replace('_75D','').replace('_',' ')
    ax.text(*xy,label,ha='center',va='center',fontsize=8.1,
            fontweight='bold', color='#24364B',zorder=7)

max_n = max(supports.values())
for category in attack_order:
    xy = attack_xy[category]
    radius = 0.10 + 0.32*np.sqrt(supports[category]/max_n)
    ax.add_patch(Circle(xy,radius,facecolor='#1D91A5',
                        edgecolor='white',lw=1.5,zorder=6))
    direction = xy/np.linalg.norm(xy)
    text_xy = xy+direction*0.50
    ax.text(*text_xy, f'{SHORT.get(category,category)}\n(n={supports[category]:,})',
            ha='left' if text_xy[0] > .2 else 'right' if text_xy[0] < -.2 else 'center',
            va='center', fontsize=9.1, color='#1B2C40', zorder=7)

ax.set_xlim(-8.1,8.1)
ax.set_ylim(-7.7,7.5)
fig.suptitle('Model–Attack Recognition Network',fontsize=20,
             fontweight='bold',color='#142B43',y=.975)
fig.text(.5,.944,'CIC development evaluation  |  Link width indicates recall  |  Node size indicates category support',
         ha='center',color='#63758A',fontsize=10.5)
fig.text(.5,.035,'Connections show measured category recall, not physical network traffic or feature similarity.',
         ha='center',color='#63758A',fontsize=9.2)
export(fig, 'Fig01_model_attack_network')
plt.show()

## 3. Attack-type recall radial map

Each ring is a model. Each section is an attack category. Darker purple marks low recall and teal marks high recall.

This makes the differences between compression methods easier to see.

In [ ]:
fig = plt.figure(figsize=(17, 12.8))
ax = fig.add_axes([.045,.07,.66,.86])
ax.set_aspect('equal')
ax.axis('off')

n_models=len(models)
n_categories=len(attack_order)
inner_r=1.52
ring_width=.39 if n_models >= 5 else .48
gap=.035
outer_r=inner_r + n_models*(ring_width+gap)
start_angle=90
step=360/n_categories
norm=colors.Normalize(0,1)
cmap=plt.get_cmap('viridis')

for j, category in enumerate(attack_order):
    mid=start_angle-j*step
    low=mid-step*.49
    high=mid+step*.49
    for i, model in enumerate(models):
        r=inner_r+i*(ring_width+gap)
        value=float(lookup.get((model,category),0))
        ax.add_patch(Wedge((0,0),r+ring_width,low,high,
                           width=ring_width,facecolor=cmap(norm(value)),
                           edgecolor='white',linewidth=1.6))
    angle=np.deg2rad(mid)
    direction=np.array([np.cos(angle),np.sin(angle)])
    point=direction*(outer_r+.10)
    size=24+140*np.sqrt(supports[category]/max_n)
    ax.scatter([point[0]],[point[1]],s=size,color='#D9A258',
               edgecolor='white',linewidth=1,zorder=10)
    label_xy=direction*(outer_r+.50)
    ax.text(*label_xy,SHORT.get(category,category),
            ha='left' if label_xy[0]>.25 else 'right' if label_xy[0]<-.25 else 'center',
            va='center',fontsize=9.6,color='#1B2C40')

ax.add_patch(Circle((0,0),inner_r-.08,fc='#F2F8FA',ec='#C7D8DF',lw=1.25))
ax.text(0,.45,'Attack-type\nrecall',ha='center',va='center',
        fontsize=18,fontweight='bold',color='#17354D')
ax.text(0,-.23,f'{n_models} model variants\n{n_categories} attack categories',
        ha='center',va='center',fontsize=10,color='#4C6678',linespacing=1.6)
lim=outer_r+2.35
ax.set_xlim(-lim,lim)
ax.set_ylim(-lim,lim)

fig.text(.72,.87,'MODEL RINGS',fontsize=12,fontweight='bold',color='#1B354C')
for i,model in enumerate(models):
    fig.text(.72,.823-i*.045,f'{i+1}  {model}',fontsize=10.2,color='#263C50')
fig.text(.72,.41,'RECALL',fontsize=12,fontweight='bold',color='#1B354C')
color_ax=fig.add_axes([.72,.365,.22,.023])
cb=plt.colorbar(cm.ScalarMappable(norm=norm,cmap=cmap),cax=color_ax,orientation='horizontal')
cb.set_ticks([0,.25,.5,.75,1])
cb.ax.tick_params(labelsize=9)
fig.text(.72,.28,'SUPPORT',fontsize=12,fontweight='bold',color='#1B354C')
fig.text(.72,.25,'Gold dots: number of attack records',fontsize=9.7,color='#52687E')
fig.text(.72,.21,'Larger dot = more records',fontsize=9.7,color='#52687E')
fig.suptitle('Attack-Type Recall Radial Map',fontsize=20,
             fontweight='bold',color='#142B43',y=.975)
fig.text(.5,.941,'CIC development evaluation  |  Ring colors show exact per-category recall',
         fontsize=10.3,color='#63758A',ha='center')
export(fig, 'Fig02_attack_recall_radial')
plt.show()

## 4. Compression trade-off

The map compares classification performance, deployment size, binary PR-AUC, and batch-256 CPU latency. All values come from the CIC development experiments.

The plot does not assume that a smaller model will always run faster.

In [ ]:
d=COMPRESSION.copy()
for col in ['Macro_F1','Binary_PR_AUC','Deployment_Artifacts_KB','Mean_Total_ms','P95_Total_ms']:
    d[col]=pd.to_numeric(d[col], errors='raise')
if d.duplicated('Model').any():
    raise ValueError('Repeated model in compression results')
if not np.isfinite(d[['Macro_F1','Binary_PR_AUC','Deployment_Artifacts_KB','P95_Total_ms']]).all().all():
    raise ValueError('Compression results contain invalid numerical values')

fig, ax = plt.subplots(figsize=(13,9.2))
fig.subplots_adjust(left=.115,right=.83,bottom=.14,top=.83)
ax.set_facecolor('#F9FBFD')
norm=colors.Normalize(d['Binary_PR_AUC'].min()-.002,d['Binary_PR_AUC'].max()+.002)
cmap=plt.get_cmap('viridis')
low=float(d['P95_Total_ms'].min()); high=float(d['P95_Total_ms'].max())
def area(t):
    return 260 + 640*(t-low)/max(high-low,1e-6)

def dominates(a,b):
    return (a['Deployment_Artifacts_KB']<=b['Deployment_Artifacts_KB'] and
            a['Macro_F1']>=b['Macro_F1'] and
            (a['Deployment_Artifacts_KB']<b['Deployment_Artifacts_KB'] or
             a['Macro_F1']>b['Macro_F1']))
front=[]
for _,b in d.iterrows():
    if not any(dominates(a,b) for _,a in d.iterrows() if a['Model']!=b['Model']):
        front.append(b)
front=sorted(front,key=lambda r:r['Deployment_Artifacts_KB'])
if len(front)>1:
    ax.plot([r['Deployment_Artifacts_KB'] for r in front],
            [r['Macro_F1'] for r in front],
            color='#7F93A6',lw=1.6,ls='--',zorder=2,label='Accuracy–size frontier')

markers=['o','s','D','^','P','X']
for i,(_,row) in enumerate(d.iterrows()):
    x=float(row['Deployment_Artifacts_KB']); y=float(row['Macro_F1'])
    clr=cmap(norm(float(row['Binary_PR_AUC'])))
    ax.scatter([x],[y],s=area(float(row['P95_Total_ms'])),
               c=[clr],marker=markers[i%len(markers)],
               edgecolors='#1E334B',lw=1.25,zorder=5)
    label=row['Model'].replace('_',' ')
    dy=25 if i%2==0 else -30
    dx=-30 if i%3==0 else 32 if i%3==1 else 68
    ax.annotate(label,(x,y),xytext=(dx,dy),textcoords='offset points',
                fontsize=9.6,ha='center',va='center',
                arrowprops=dict(arrowstyle='-',color='#7891A5',lw=.7),
                bbox=dict(boxstyle='round,pad=.25',fc='white',ec='#DCE5ED',alpha=.95))
ax.set_xlabel('Deployment artifact size (KiB) — smaller is better')
ax.set_ylabel('Multiclass macro-F1 — higher is better')
ax.grid(alpha=.35,color='#9AAFC2',zorder=0)
ax.margins(x=.28,y=.35)
ax.set_title('Compression Accuracy–Storage Trade-off',loc='left',
             fontsize=18,fontweight='bold',color='#142B43',pad=26)
ax.text(0,1.025,'CIC development set  |  Batch-256 CPU benchmark',
        transform=ax.transAxes,fontsize=10,color='#60768A')
cax=fig.add_axes([.86,.26,.025,.47])
cb=plt.colorbar(cm.ScalarMappable(norm=norm,cmap=cmap),cax=cax)
cb.set_label('Binary PR-AUC',fontsize=10)
fig.text(.855,.18,f'P95 latency\n{low:.1f}–{high:.1f} ms\n(bubble size)',
         fontsize=10,color='#4C6278',ha='left')
fig.text(.11,.045,'Storage includes serialized encoder, embedding scaler, adapted classifier, and source scaler.\n'
         'Measured CPU time varies by environment; all accuracy values are from CIC development.',
         fontsize=9,color='#6B7D90',linespacing=1.5)
export(fig,'Fig03_compression_tradeoff')
plt.show()

## 5. Adaptation across three seeds

Each line shows one experimental seed. The panels compare multiclass macro-F1 and binary PR-AUC for the three label-selection methods.

In [ ]:
data=ADAPTATION.copy()
labels={'Random':'Random','Standard_RL':'Standard RL','Diversity_Aware_RL':'Diversity-aware RL'}
methods=[name for name in ['Random','Standard_RL','Diversity_Aware_RL'] if name in set(data['Method'])]
other=sorted(set(data['Method'])-set(methods))
methods+=other
seeds=sorted(data['Seed'].unique())
if data.duplicated(['Seed','Method']).any():
    raise ValueError('Repeated adaptation result for a seed and method')
if len(seeds)<2 or len(methods)<2:
    raise ValueError('At least two seeds and two methods are needed')

fig,axes=plt.subplots(1,2,figsize=(15.5,6.8))
fig.subplots_adjust(left=.085,right=.97,bottom=.21,top=.82,wspace=.24)
metrics=[('Macro_F1','Multiclass macro-F1'),('Binary_PR_AUC','Binary PR-AUC')]
seed_colors=plt.cm.tab10(np.linspace(.04,.7,len(seeds)))
for ax,(metric,caption) in zip(axes,metrics):
    table=data.pivot(index='Seed',columns='Method',values=metric)
    for i,seed in enumerate(seeds):
        y=table.loc[seed,methods].to_numpy(dtype=float)
        ax.plot(range(len(methods)),y,marker='o',ms=8,linewidth=1.7,
                color=seed_colors[i],alpha=.78,label=f'Seed {seed}')
    mean=table[methods].mean(axis=0).to_numpy(dtype=float)
    ax.plot(range(len(methods)),mean,marker='D',ms=9,lw=3,
            color='#163D5B',label='Mean',zorder=6)
    for i,y in enumerate(mean):
        ax.text(i,y+0.015 if metric=='Macro_F1' else y+.001,
                f'{y:.3f}',ha='center',fontsize=9.5,
                fontweight='bold',color='#163D5B')
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([labels.get(m,m) for m in methods],fontsize=10)
    ax.set_xlim(-.30,len(methods)-.70)
    values=table[methods].to_numpy(dtype=float).ravel()
    p=max(values)-min(values)
    pad=max(.03 if metric=='Macro_F1' else .006,p*.23)
    ax.set_ylim(min(values)-pad,max(values)+pad)
    ax.set_ylabel(caption)
    ax.set_facecolor('#FAFCFE')
    ax.grid(alpha=.35,color='#9AAFC2')
    ax.set_title(caption,loc='left',fontweight='bold',color='#142B43',pad=15)
axes[0].legend(frameon=False,loc='lower right',fontsize=9)
fig.suptitle('Target Adaptation Across Three Seeds',fontsize=19,
             fontweight='bold',color='#142B43',y=.97)
fig.text(.5,.91,'Same target-label budget for each method  |  CIC development evaluation',
         ha='center',fontsize=10.5,color='#60768A')
fig.text(.09,.075,'Lines connect methods within each seed; the dark line shows the mean.',
         fontsize=9.5,color='#63758A')
export(fig,'Fig04_adaptation_three_seeds')
plt.show()

## 6. Frozen holdout attack recognition

This diagram shows recall for each attack category on the separate frozen CIC holdout. Distance from the center represents recall, and point size represents the number of attack records.

The holdout contains unseen records from the CIC corpus. It is not an independent traffic-collection environment.

In [ ]:
h=HOLDOUT_CATEGORY.copy()
h['Recall']=pd.to_numeric(h['Recall'])
h['Test_Records']=pd.to_numeric(h['Test_Records'])
h=h.sort_values('Attack_Category').reset_index(drop=True)
if not h['Recall'].between(0,1).all():
    raise ValueError('Invalid holdout recall')

fig=plt.figure(figsize=(15,10.5))
ax=fig.add_axes([.12,.08,.70,.78],projection='polar')
n=len(h)
angles=np.pi/2-2*np.pi*np.arange(n)/n
ax.set_theta_direction(1)
ax.set_ylim(0,1.13)
ax.set_yticks([.25,.5,.75,1])
ax.set_yticklabels(['0.25','0.50','0.75','1.00'],color='#8FA1B4')
ax.set_xticks(angles)
ax.set_xticklabels([SHORT.get(v,v) for v in h['Attack_Category']],fontsize=9)
ax.tick_params(axis='x',pad=16)
ax.grid(color='#D7E2EB',alpha=.65)
ax.spines['polar'].set_color('#C6D6E1')
count=h['Test_Records'].to_numpy(dtype=float)
recall=h['Recall'].to_numpy(dtype=float)
colors_scatter=cm.viridis(recall)
scales=65+680*np.sqrt(count/np.maximum(count.max(),1))
for angle,rec in zip(angles,recall):
    ax.plot([angle,angle],[0,max(rec,0.01)],color='#8EA7B9',lw=1.5,alpha=.55,zorder=2)
ax.scatter(angles,recall,s=scales,color=colors_scatter,
           edgecolors='white',linewidth=1.2,zorder=5)
ax.set_title('Frozen Model: Attack-Type Recall',fontsize=18,
             fontweight='bold',color='#142B43',pad=45)
fig.text(.5,.96,'20,000 unused CIC records  |  2,417 attacks  |  No classifier refitting',
         ha='center',color='#63758A',fontsize=10.5)
fig.text(.79,.78,'HOLDOUT METRICS',fontweight='bold',fontsize=11,color='#142B43')
summary=[
    ('Macro-F1',HOLDOUT.get('Multiclass_Macro_F1')),
    ('Binary PR-AUC',HOLDOUT.get('Binary_PR_AUC')),
    ('Exact attack type',HOLDOUT.get('Exact_Attack_Type_Accuracy')),
    ('Precision @ budget',HOLDOUT.get('Precision_At_Budget')),
    ('Recall @ budget',HOLDOUT.get('Recall_At_Budget')),
]
for i,(label,val) in enumerate(summary):
    fig.text(.79,.735-i*.07,f'{label}\n{float(val):.4f}',
             fontsize=10.7,color='#344C60',linespacing=1.35)
fig.text(.12,.035,'Point size: category support. Color and radial position: recall. Rare attack types may have little or no correct detection.',
         fontsize=9.4,color='#60768A')
export(fig,'Fig05_frozen_holdout_recall')
plt.show()

## 7. Saved figures

All figures were generated from the numerical result files. Development comparisons and frozen holdout results are presented separately.

In [ ]:
for filename in sorted(OUT.glob('Fig0*.png')):
    print(filename.name)
print('Figure files:', OUT)